# COMP5329 — Week 6 Self-Study Material
## Graph Convolutional Networks: Complete Reference

**Scope.** Lecture-complete reference for Week 6. Covers every topic on the lecture slides with (a) mathematical explanation, (b) PyTorch / NumPy implementation where applicable, and (c) numerical verification on concrete graphs. Assumes undergraduate linear algebra (eigenvalues, eigenvectors, matrix decomposition) and basic PyTorch.

**Relationship to the tutorial notebook.** This is a standalone document. The tutorial notebook (`Week6_Graph_Convolutional_Networks.ipynb`) is a focused 60-minute session; this notebook is the full reference, including the spectral approach, algebraic derivations, and prediction tasks and MCP material that were dropped from the tutorial for time.


## Table of Contents

**Part I — Motivation & Background**
- 0.1 Grid-structured data vs graph-structured data
- 0.2 Why CNNs and RNNs fail on graphs

**Part II — Graph Representations**
- 1.1 Formal definition of a graph
- 1.2 Three edge representations: edge list, adjacency matrix, incidence matrix
- 1.3 Variants: weighted, directed, with self-loops
- 1.4 Degree matrix
- 1.5 Dense vs sparse; connected components
- 1.6 Notation cheat-sheet

**Part III — Spatial Approach (Kipf & Welling GCN)**
- 2.1 CNN convolution as weighted message aggregation
- 2.2 Key idea 1: weight sharing on graphs
- 2.3 From raw A to symmetric normalization — four steps
- 2.4 Algebraic derivation of the symmetric normalized form
- 2.5 GCNLayer in PyTorch
- 2.6 Semi-supervised node classification on Karate Club
- 2.7 The Kipf & Welling architecture

**Part IV — Spectral Approach**
- 3.1 Graph Laplacian L = D − A
- 3.2 Normalized Laplacian and its eigenspectrum
- 3.3 The convolution theorem
- 3.4 Classical Fourier transform recap
- 3.5 Fourier transform on graphs
- 3.6 Spectral graph convolution
- 3.7 Spectral GCN Version 1.0 (Bruna et al. 2013)
- 3.8 Spectral GCN Version 2.0 — ChebNet (Defferrard et al. 2016)
- 3.9 Chebyshev polynomial recursion
- 3.10 Bridge: from ChebNet K=1 to Kipf & Welling spatial GCN

**Part V — Prediction Tasks Beyond Node Classification**
- 4.1 Graph classification with global readout
- 4.2 Link prediction with dot-product scoring

**Part VI — Claude Code and MCP (Model Context Protocol)**
- 5.1 What is MCP
- 5.2 MCP vs Skills
- 5.3 MCP server primitives
- 5.4 Using MCP in Claude Code
- 5.5 Popular MCP servers
- 5.6 Command quick-reference


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)


---
## Part I · Motivation & Background

### 0.1 Grid-structured data vs graph-structured data

Most successful deep-learning architectures were designed for **Euclidean, grid-structured** inputs:

| Domain | Structure | Architecture |
|---|---|---|
| Images | 2-D pixel grid | CNN |
| Speech / time series | 1-D sequence | RNN, 1-D CNN |
| Game boards (Go, chess) | 2-D grid | CNN |

The defining property of a grid is that **every position has the same, ordered local neighbourhood**: a pixel in a 2-D image has exactly 8 neighbours in fixed positions (N, NE, E, SE, S, SW, W, NW). Convolutions exploit this by assigning a distinct learnable weight to each position in a $3 \times 3$ kernel.

A large class of real-world data lives on **irregular relational structures** — graphs:

| Domain | Nodes | Edges |
|---|---|---|
| Social networks | Users | Friendships |
| Citation networks | Papers | Cites |
| Communication networks | Devices | Links |
| Multi-agent systems | Agents | Interactions |
| Molecules | Atoms | Bonds |
| Protein interaction networks | Proteins | Binding |

### 0.2 Why CNNs and RNNs fail on graphs

Two structural differences break convolutional architectures on graphs:

1. **Variable neighbourhood size.** On a 2-D image every pixel has 8 neighbours (4 on the edges, 3 at the corners). On a graph a node can have anywhere from 0 to $N-1$ neighbours. A $3 \times 3$ kernel is a single tensor of 9 weights — there is no analogue for a graph node whose neighbour count is unknown in advance.
2. **No canonical ordering of neighbours.** On an image the 8 neighbours have a spatial ordering (N, NE, ...), so a kernel can learn a distinct weight per position. On a graph the neighbour set $\mathcal{N}(i)$ is an **unordered set** — permuting the labels of the neighbours must not change the output. CNN weights lose their meaning the moment the neighbours are unordered.

The required architecture must therefore be (a) **invariant to neighbour permutations** and (b) capable of handling **variable-size neighbourhoods**. GCN achieves both by applying the **same** linear transform to every neighbour and then aggregating with a permutation-symmetric operator (sum or mean).


In [ ]:
# Illustration: variable degree in a real graph
G = nx.karate_club_graph()
degrees = [d for _, d in G.degree()]

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(degrees, bins=range(min(degrees), max(degrees) + 2),
        color='#5B8DB8', edgecolor='black')
ax.set_xlabel('degree')
ax.set_ylabel('# nodes')
ax.set_title(f"Karate Club: node degrees range from {min(degrees)} to {max(degrees)}")
plt.tight_layout(); plt.show()


---
## Part II · Graph Representations

### 1.1 Formal definition

A graph $\mathcal{G} = (\mathcal{V}, \mathcal{E})$ consists of

- a **node set** $\mathcal{V} = \{v_i \mid i = 1, \ldots, N\}$ of cardinality $|\mathcal{V}| = N$;
- an **edge set** $\mathcal{E} = \{e_{ij} \mid v_i \text{ is connected to } v_j\}$.

Nodes may carry features: $X \in \mathbb{R}^{N \times d}$ is the **node attribute matrix**, where row $i$ is the $d$-dimensional feature vector of node $v_i$.

### 1.2 Three edge representations

Three data structures are commonly used to describe $\mathcal{E}$:

1. **Edge list** — a list of tuples $(i, j)$. Space $O(|\mathcal{E}|)$. Best for sparse graphs and for frameworks that expect `edge_index` (PyG, DGL).
2. **Adjacency matrix** $A \in \mathbb{R}^{N \times N}$ — $A_{ij} = 1$ if $(v_i, v_j) \in \mathcal{E}$, else $0$. Space $O(N^2)$. Best for dense graphs and for matrix-algebra formulations (our case).
3. **Incidence matrix** $M \in \mathbb{R}^{N \times |\mathcal{E}|}$ — $M_{ik} = 1$ if node $i$ is an endpoint of edge $e_k$ (or $\pm 1$ for a directed edge, encoding source/target). Space $O(N |\mathcal{E}|)$. Rarely used directly in GNNs but essential for the graph Laplacian view in Part IV.

The small example graph used throughout the lecture has 5 nodes and 8 edges.


In [ ]:
# 5-node example graph (lecture pages 8-14)
nodes = ['a', 'b', 'c', 'd', 'e']
edges = [('a','b'), ('a','d'), ('a','e'),
         ('b','c'), ('b','d'), ('b','e'),
         ('c','d'),
         ('d','e')]
idx = {n: i for i, n in enumerate(nodes)}
N = len(nodes)

# --- 1. Edge list ---
print(f"Edge list ({len(edges)} edges):")
for u, v in edges:
    print(f"  ({u}, {v})")

# --- 2. Adjacency matrix ---
A = np.zeros((N, N), dtype=int)
for u, v in edges:
    A[idx[u], idx[v]] = 1
    A[idx[v], idx[u]] = 1
print("\nAdjacency matrix A:")
print("     " + "  ".join(nodes))
for i, row in enumerate(A):
    print(f"  {nodes[i]}  {row}")

# --- 3. Incidence matrix ---
M = np.zeros((N, len(edges)), dtype=int)
for k, (u, v) in enumerate(edges):
    M[idx[u], k] = 1
    M[idx[v], k] = 1
print("\nIncidence matrix M (nodes x edges):")
print("     " + " ".join(f'e{k+1}' for k in range(len(edges))))
for i, row in enumerate(M):
    print(f"  {nodes[i]}   " + "  ".join(str(x) for x in row))


### 1.3 Adjacency matrix variants

**Adjacency with self-connections** $\hat{A} = A + I_N$: adds a self-loop at every node. This form appears in the GCN propagation rule, because it guarantees that a node's own features are part of its own neighbourhood — otherwise repeated convolution would erase them.

**Weighted adjacency** $A^W$: entries take real values $w_{ij}$ rather than $\{0, 1\}$. Used when edges carry strengths (chemical bond order, friendship intensity, communication bandwidth).

**Directed graph** adjacency: $A$ need not be symmetric. $A_{ij} = 1$ means an edge from $i$ *to* $j$. The symmetric GCN rule does not directly apply — directed graphs usually require message-passing formulations with separate incoming / outgoing aggregations.


In [ ]:
# Variants on the 5-node example
I_N = np.eye(N, dtype=int)
A_hat = A + I_N
print("A_hat = A + I_N  (adjacency with self-loops):")
print(A_hat)

# Weighted variant
A_weighted = np.zeros_like(A, dtype=float)
for k, (u, v) in enumerate(edges):
    w = 0.5 + 0.1 * k
    A_weighted[idx[u], idx[v]] = w
    A_weighted[idx[v], idx[u]] = w
print("\nWeighted adjacency A^W (entries are real numbers):")
print(np.round(A_weighted, 2))

# Directed variant
A_dir = np.zeros_like(A)
for u, v in edges:
    if u < v:
        A_dir[idx[u], idx[v]] = 1
print("\nDirected adjacency (orient every edge small->large, asymmetric):")
print(A_dir)
print("symmetric?", np.array_equal(A_dir, A_dir.T))


### 1.4 Degree matrix

The **degree** of node $i$ is the number of edges incident to it:
$$d_i = \sum_{j=1}^{N} A_{ij}.$$
The **degree matrix** $D$ is the diagonal matrix with $D_{ii} = d_i$.

For the self-loop version we use $\hat{D}$ — the degree matrix of $\hat{A}$. Since each self-loop adds 1 to the corresponding diagonal entry, $\hat{D} = D + I_N$ for a simple unweighted graph.


In [ ]:
d     = A.sum(axis=1)
d_hat = A_hat.sum(axis=1)
D      = np.diag(d)
D_hat  = np.diag(d_hat)

print("Degree vector d    :", d,
      "(sum =", int(d.sum()), "= 2 x #edges =", 2 * len(edges), ")")
print("Degree vector d_hat:", d_hat,
      "(each = d_i + 1 because of the self-loop)")
print("\nD =")
print(D)
print("\nD_hat =")
print(D_hat)


### 1.5 Basic properties

**Dense graph**: $|\mathcal{E}| = \Theta(N^2)$ — a significant fraction of the $\binom{N}{2}$ possible pairs are edges. Storing as adjacency matrix is fine.

**Sparse graph**: $|\mathcal{E}| = \Theta(N)$ — each node has a bounded number of neighbours independent of $N$. Adjacency matrix storage wastes memory; edge lists or sparse-matrix representations are preferred. Most real-world graphs are sparse.

**Directed vs undirected**: an edge $(i, j)$ in an undirected graph corresponds to both $A_{ij} = 1$ and $A_{ji} = 1$; a directed graph distinguishes the two.

**Connected component**: a maximal subgraph in which every pair of nodes is joined by at least one path. A graph partitions uniquely into connected components, and this partition is readable from the rank structure of the Laplacian (Part IV §3.1): the multiplicity of the eigenvalue $0$ of $L$ equals the number of connected components.


In [ ]:
# Build a deliberately disconnected graph
G_disc = nx.Graph()
G_disc.add_nodes_from(range(8))
G_disc.add_edges_from([(0, 1), (1, 2),           # component 1
                       (3, 4),                    # component 2
                       (5, 6), (6, 7), (5, 7)])   # component 3
components = list(nx.connected_components(G_disc))
print(f"#components = {len(components)}")
for k, comp in enumerate(components):
    print(f"  component {k}: {sorted(comp)}")

# Verify via Laplacian eigenvalues
A_d = nx.to_numpy_array(G_disc)
L_d = np.diag(A_d.sum(1)) - A_d
eigvals_L = np.sort(np.linalg.eigvalsh(L_d))
print(f"\nLaplacian eigenvalues (ascending): {np.round(eigvals_L, 4)}")
print(f"#zero eigenvalues = {int(np.isclose(eigvals_L, 0, atol=1e-8).sum())}  "
      f"(matches #components)")


### 1.6 Notation cheat-sheet

| Symbol | Shape | Meaning |
|---|---|---|
| $\mathcal{G} = (\mathcal{V}, \mathcal{E})$ | — | Graph with node set and edge set |
| $N = \|\mathcal{V}\|$ | scalar | Number of nodes |
| $X$ | $N \times d$ | Node attribute matrix |
| $H^{(l)}$ | $N \times F_l$ | Hidden representation at layer $l$ ($H^{(0)} = X$) |
| $A$ | $N \times N$ | Adjacency matrix |
| $I_N$ | $N \times N$ | Identity matrix of order $N$ |
| $\hat{A} = A + I_N$ | $N \times N$ | Adjacency with self-loops |
| $D$ | $N \times N$ | Degree matrix (diagonal, $D_{ii} = \sum_j A_{ij}$) |
| $\hat{D}$ | $N \times N$ | Degree matrix of $\hat{A}$ |
| $M$ | $N \times \|\mathcal{E}\|$ | Incidence matrix |
| $L = D - A$ | $N \times N$ | Unnormalized graph Laplacian |
| $L' = I_N - D^{-1/2} A D^{-1/2}$ | $N \times N$ | Symmetrically normalized Laplacian |
| $\tilde{L} = \frac{2}{\lambda_\max} L' - I_N$ | $N \times N$ | Laplacian rescaled to $[-1, 1]$ (used in ChebNet) |
| $U, \Lambda$ | $N \times N$ | Eigenvectors / eigenvalues of the Laplacian |
| $W^{(l)}$ | $F_l \times F_{l+1}$ | Learnable weight matrix at layer $l$ |


---
## Part III · Spatial Approach (Kipf & Welling GCN)

### 2.1 CNN convolution as weighted message aggregation

A single 2-D convolutional layer with a $3 \times 3$ filter updates a pixel $h_4$ (the centre of its $3 \times 3$ window) by:
$$h_4^{(l+1)} = \sigma\!\left(W_0^{(l)} h_0^{(l)} + W_1^{(l)} h_1^{(l)} + \ldots + W_8^{(l)} h_8^{(l)}\right),$$
where $h_0, \ldots, h_8$ are the 9 pixels in the window. This formula performs two operations:

1. **Transform every neighbour individually**: $W_k h_k$. Each of the 9 positions gets its own weight matrix because the grid has a canonical ordering.
2. **Sum up all transformed messages**: $\sum_k W_k h_k$.

This is a message-passing interpretation: each neighbour sends a learned message and the centre aggregates them. GCN keeps step 2 identical but changes step 1 — since a graph has no canonical ordering, GCN cannot assign a different $W_k$ to each neighbour.


In [ ]:
# 3x3 convolution as explicit weighted sum
rng = np.random.default_rng(0)
img    = rng.normal(size=(5, 5))
kernel = rng.normal(size=(3, 3))

def conv3x3(img, kernel):
    H, W = img.shape
    out = np.zeros((H - 2, W - 2))
    for i in range(H - 2):
        for j in range(W - 2):
            patch = img[i:i+3, j:j+3]
            out[i, j] = np.sum(patch * kernel)   # weighted sum over 9 positions
    return out

print("5x5 image convolved with a 3x3 kernel -> 3x3 output:")
print(np.round(conv3x3(img, kernel), 3))


### 2.2 Key idea 1 — weight sharing in graph convolution

For a graph, the lecture proposes:
$$h_i^{(l+1)} = \sigma\!\left(h_i^{(l)} W_0^{(l)} \;+\; \sum_{j \in \mathcal{N}(i)} \frac{1}{c_{ij}} h_j^{(l)} W_1^{(l)}\right),$$
where

- $W_0^{(l)}$ is the **self weight** applied to node $i$'s own features,
- $W_1^{(l)}$ is the **neighbour weight**, shared across *all* neighbours $j \in \mathcal{N}(i)$,
- $c_{ij}$ is a normalization constant (fixed or trainable).

Only **two** weight matrices are learned per layer, regardless of the degree of node $i$. This is the key trick: since the neighbours are unordered, there is no basis on which to assign them different weights, so they all share $W_1$. The only distinction the model can make is "self" vs "neighbour".

In the Kipf & Welling form, $W_0$ and $W_1$ are further tied together — $\hat{A} = A + I_N$ folds the self-loop into the neighbourhood, and a single $W$ matrix handles both:
$$H^{(l+1)} = \sigma\!\left(\hat{D}^{-1/2} \hat{A} \hat{D}^{-1/2} \,H^{(l)} W^{(l)}\right).$$

**Desirable properties** of this weight sharing (lecture page 24):

- **Weight sharing over all locations.** The same $W$ is used at every node; parameter count is $O(F_\text{in} \times F_\text{out})$ per layer, independent of $N$.
- **Invariance to permutations.** If the nodes are relabelled by a permutation $P$, the output transforms covariantly as $PH^{(l+1)}$. Model output on a given node does not depend on how nodes are numbered.
- **Linear complexity** $O(|\mathcal{E}|)$. The matrix–vector product $\hat{A} h$ touches only existing edges when $\hat{A}$ is stored sparsely.
- **Applicable in transductive and inductive settings.** Transductive = trained and evaluated on the same graph (e.g., Karate Club); inductive = generalises to unseen graphs at test time (e.g., a GCN trained on one molecule predicts properties of new molecules). GCN supports both because $W$ is independent of graph size.

**Limitations** (lecture page 24):

- **Requires gating / residual connections to go deep.** Plain GCN layers beyond $\sim 3$ suffer from *over-smoothing*: repeated multiplication by $\hat{D}^{-1/2} \hat{A} \hat{D}^{-1/2}$ drives all node representations toward the dominant eigenvector of the propagation matrix, washing out class distinctions.
- **Indirect support for edge features.** The vanilla GCN only uses the binary, symmetric-normalized adjacency; edge attributes (bond order, timestamp) must be injected through separate mechanisms (Message Passing Neural Networks, GAT, R-GCN).


### 2.3 Building the propagation rule step by step

The lecture constructs $\hat{D}^{-1/2}\hat{A}\hat{D}^{-1/2}$ in four stages. We reproduce each stage on the 5-node example applied to a single feature vector $h$.

**Step 1 — raw adjacency.** $Y = A h$. Row $i$ of $Y$ is $\sum_{j \in \mathcal{N}(i)} h_j$: a sum over *neighbours only*. The central node's own feature is discarded. After one layer, a node's feature is entirely determined by its neighbours — its own history is gone.

**Step 2 — self-loop.** $Y = \hat{A} h$ with $\hat{A} = A + I_N$. Each row now sums over **self + neighbours**, preserving the node's own feature.

**Step 3 — row normalization.** $Y = \hat{D}^{-1} \hat{A} h$. Each row of $\hat{D}^{-1} \hat{A}$ has equal entries $1/\hat{d}_i$ on the neighbourhood and $0$ elsewhere, so $Y_i$ is the **unweighted mean** of $h$ over the self-including neighbourhood. Magnitudes no longer scale with degree.

**Step 4 — symmetric normalization.** $Y = \hat{D}^{-1/2} \hat{A} \hat{D}^{-1/2} h$. Each non-zero entry becomes $1 / \sqrt{\hat{d}_i \hat{d}_j}$, splitting the $1/\hat{d}$ factor between sender and receiver. The resulting matrix is **symmetric** (important for the spectral view in Part IV) and damps contributions from high-degree neighbours more than row normalization does.


In [ ]:
# Compare all four propagation operators on the 5-node example
h = np.array([1.0, 2.0, 3.0, 4.0, 5.0])   # dummy scalar feature per node

Y1 = A @ h                                              # Step 1 — raw A
Y2 = A_hat @ h                                          # Step 2 — with self-loops

D_hat_inv = np.diag(1.0 / d_hat)
S_row     = D_hat_inv @ A_hat
Y3 = S_row @ h                                          # Step 3 — row normalized

D_hat_invsqrt = np.diag(d_hat ** -0.5)
S_sym         = D_hat_invsqrt @ A_hat @ D_hat_invsqrt
Y4 = S_sym @ h                                          # Step 4 — symmetric normalized

print("Input h :", h)
print()
print("Step 1  A h                     :", np.round(Y1, 4))
print("Step 2  A_hat h                 :", np.round(Y2, 4))
print("Step 3  D_hat^-1 A_hat h        :", np.round(Y3, 4))
print("Step 4  D_hat^-1/2 A_hat D_hat^-1/2 h:", np.round(Y4, 4))

print("\nStep-3 operator symmetric? ", np.allclose(S_row, S_row.T))
print("Step-4 operator symmetric? ", np.allclose(S_sym, S_sym.T))


### 2.4 Algebraic derivation of the symmetric form

Starting from the product $\hat{D}^{-1/2} \hat{A} \hat{D}^{-1/2} H$, we show that row $i$ can be written as a degree-weighted sum over neighbours. This is the derivation on lecture page 29.

$$
\begin{aligned}
\bigl[\hat{D}^{-1/2} \hat{A} \hat{D}^{-1/2} H\bigr]_i
  &= \sum_k \bigl[\hat{D}^{-1/2}\bigr]_{ik} \bigl[\hat{A} \hat{D}^{-1/2} H\bigr]_k && \text{(matrix multiply)} \\
  &= \bigl[\hat{D}^{-1/2}\bigr]_{ii} \bigl[\hat{A} \hat{D}^{-1/2} H\bigr]_i && \text{(diagonal: only $k=i$ contributes)} \\
  &= \hat{D}_{ii}^{-1/2} \sum_j \hat{A}_{ij} \bigl[\hat{D}^{-1/2} H\bigr]_j \\
  &= \hat{D}_{ii}^{-1/2} \sum_j \hat{A}_{ij} \sum_k \bigl[\hat{D}^{-1/2}\bigr]_{jk} H_k \\
  &= \hat{D}_{ii}^{-1/2} \sum_j \hat{A}_{ij} \hat{D}_{jj}^{-1/2} H_j && \text{(diagonal again)} \\
  &= \sum_j \frac{\hat{A}_{ij}}{\sqrt{\hat{D}_{ii} \hat{D}_{jj}}} H_j.
\end{aligned}
$$

Interpretation: the updated representation of node $i$ is a sum over its (self-including) neighbours $j$, where each $j$'s feature is weighted by $1 / \sqrt{\hat{d}_i \hat{d}_j}$ — the **geometric mean** of the two endpoints' inverse square-root degrees. Because this formula is symmetric in $i$ and $j$, information flow between $i$ and $j$ is **reciprocal**: what $i$ receives from $j$ equals what $j$ receives from $i$.

Numerical verification of the row-wise summation form:


In [ ]:
# Verify the sum-form matches the matrix-form
lhs = S_sym @ h

rhs = np.zeros(N)
for i in range(N):
    for j in range(N):
        rhs[i] += A_hat[i, j] / np.sqrt(d_hat[i] * d_hat[j]) * h[j]

print("matrix form    :", np.round(lhs, 6))
print("summation form :", np.round(rhs, 6))
print("max diff       :", np.abs(lhs - rhs).max())


### 2.5 `GCNLayer` — PyTorch implementation

The full GCN propagation rule packaged as an `nn.Module`:
$$H^{(l+1)} = \sigma\!\left(\hat{A}_\text{norm}\, H^{(l)}\, W^{(l)}\right).$$

The weight matrix $W$ is the only learnable tensor. The normalized adjacency $\hat{A}_\text{norm}$ is a pre-computed, fixed tensor — it does **not** carry gradients. The backward pass is delegated entirely to PyTorch's autograd.


In [ ]:
def build_A_hat_norm(A: np.ndarray) -> torch.Tensor:
    '''Return D_hat^{-1/2} (A + I) D_hat^{-1/2} as a float32 torch tensor.'''
    n       = A.shape[0]
    A_hat   = A + np.eye(n)
    d_hat   = A_hat.sum(axis=1)
    d_inv   = np.power(d_hat, -0.5)
    d_inv[np.isinf(d_inv)] = 0.0
    D_inv   = np.diag(d_inv)
    A_norm  = D_inv @ A_hat @ D_inv
    return torch.tensor(A_norm, dtype=torch.float32)


class GCNLayer(nn.Module):
    '''Single Graph Convolution layer:  H_out = sigma( A_hat_norm @ H @ W ).'''

    def __init__(self, in_features: int, out_features: int,
                 activation: bool = True):
        super().__init__()
        self.W          = nn.Parameter(torch.empty(in_features, out_features))
        self.activation = activation
        nn.init.xavier_uniform_(self.W)

    def forward(self, H: torch.Tensor,
                A_hat_norm: torch.Tensor) -> torch.Tensor:
        out = A_hat_norm @ H @ self.W
        if self.activation:
            out = F.relu(out)
        return out


### 2.6 Semi-supervised node classification on Karate Club

Zachary's Karate Club is a real social network: 34 members, 78 friendships. In 1970 the club split into two factions (led by "Mr. Hi" and "Officer"), and the split is the ground-truth community label. The task: predict each member's faction given the graph structure, but with **only two labelled examples** at training time (node 0 = Mr. Hi, node 33 = Officer). All 34 nodes are evaluated.

This is **semi-supervised node classification**: most nodes are unlabelled, and the GCN propagates label information through the edges to reach them.


In [ ]:
# Load Karate Club and visualize the ground-truth split
G_kc  = nx.karate_club_graph()
n     = G_kc.number_of_nodes()
A_kc  = nx.to_numpy_array(G_kc)

club     = nx.get_node_attributes(G_kc, 'club')
node_col = ['#E07B54' if club[i] == 'Mr. Hi' else '#5B8DB8' for i in G_kc.nodes()]
pos      = nx.spring_layout(G_kc, seed=42)

fig, ax = plt.subplots(figsize=(7, 4.5))
nx.draw_networkx(G_kc, pos=pos, node_color=node_col, node_size=400,
                 font_size=8, edge_color='#cccccc', with_labels=True, ax=ax)
ax.set_title("Zachary's Karate Club  (orange = Mr. Hi, blue = Officer)")
ax.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# Prepare tensors
A_hat_norm_kc = build_A_hat_norm(A_kc)                  # (34, 34), fixed
X_kc          = torch.eye(n, dtype=torch.float32)       # (34, 34) one-hot identity

label_map   = {'Mr. Hi': 0, 'Officer': 1}
y_kc        = torch.tensor([label_map[club[i]] for i in G_kc.nodes()])
mask_kc     = torch.zeros(n, dtype=torch.bool)
mask_kc[0]  = True
mask_kc[33] = True
print(f"Labelled nodes: {mask_kc.nonzero().squeeze().tolist()}")


In [ ]:
class GCN(nn.Module):
    '''Two-layer GCN for node classification.'''

    def __init__(self, in_features, hidden, num_classes):
        super().__init__()
        self.gcn1 = GCNLayer(in_features, hidden,     activation=True)
        self.gcn2 = GCNLayer(hidden,      num_classes, activation=False)

    def forward(self, H, A_norm):
        H = self.gcn1(H, A_norm)
        H = self.gcn2(H, A_norm)
        return H

    def embed(self, H, A_norm):
        with torch.no_grad():
            return self.gcn1(H, A_norm)


torch.manual_seed(42)
model = GCN(in_features=n, hidden=16, num_classes=2)
opt   = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

losses = []
for epoch in range(300):
    model.train()
    opt.zero_grad()
    logits = model(X_kc, A_hat_norm_kc)
    loss   = F.cross_entropy(logits[mask_kc], y_kc[mask_kc])
    loss.backward()
    opt.step()
    losses.append(loss.item())

with torch.no_grad():
    preds = model(X_kc, A_hat_norm_kc).argmax(1)
    acc   = (preds == y_kc).float().mean().item()

print(f"Accuracy on all 34 nodes (only 2 labelled during training): {acc:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(losses, color='steelblue')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('cross-entropy loss')
axes[0].set_title('Training loss (2 labelled nodes)')

palette  = ['#E07B54', '#5B8DB8']
pred_col = [palette[p.item()] for p in preds]
nx.draw_networkx(G_kc, pos=pos, node_color=pred_col, node_size=400,
                 font_size=8, edge_color='#cccccc', with_labels=True, ax=axes[1])
axes[1].set_title(f'GCN predictions  (accuracy {acc:.0%})')
axes[1].axis('off')
plt.tight_layout(); plt.show()


### 2.7 The Kipf & Welling architecture

Kipf and Welling (ICLR 2017) published the canonical 2-layer GCN for semi-supervised classification. Its structure, using the notation above:

$$
\begin{aligned}
H^{(0)} &= X \\
H^{(1)} &= \operatorname{ReLU}\!\bigl(\hat{A}_{\text{norm}}\, H^{(0)}\, W^{(0)}\bigr) \\
Z &= H^{(2)} = \hat{A}_{\text{norm}}\, H^{(1)}\, W^{(1)} \\
\text{class probabilities} &= \operatorname{softmax}(Z)
\end{aligned}
$$

Key hyperparameters:

- **Two layers** give a 2-hop receptive field: each node's logits depend on all nodes within graph distance 2. Stacking more layers extends the receptive field but triggers over-smoothing.
- **Hidden width** 16 is enough for Karate Club; larger benchmarks (Cora, Citeseer, PubMed) use 16–64.
- **Loss** is cross-entropy on the labelled subset only:
  $$\mathcal{L} = -\!\!\sum_{i \in \mathcal{Y}_L}\!\!\sum_c y_{ic} \log\!\bigl(\operatorname{softmax}(Z_i)_c\bigr).$$
  Gradients still flow through the unlabelled nodes because they participate in the forward convolution $\hat{A}_\text{norm} H^{(l)}$.


---
## Part IV · Spectral Approach

The spatial view in Part III comes from one of two independent derivations of GCN. The other derivation — historically older — is **spectral graph theory**, which defines graph convolution through the graph Laplacian and its eigendecomposition. In this part we build the spectral machinery, derive two historical spectral GCNs (Bruna 2013, Defferrard 2016), and recover the Kipf & Welling spatial rule as a special case.

### 3.1 Graph Laplacian

For an undirected graph with adjacency $A$ and degree matrix $D$, the **unnormalized graph Laplacian** is
$$L = D - A.$$

Properties:

- **Symmetric** (both $D$ and $A$ are symmetric for undirected graphs).
- **Positive semi-definite**: for any $x \in \mathbb{R}^N$,
  $$x^\top L x = \sum_{(i,j) \in \mathcal{E}} (x_i - x_j)^2 \;\ge\; 0.$$
  So $L$ has non-negative real eigenvalues $0 = \lambda_0 \le \lambda_1 \le \ldots \le \lambda_{N-1}$.
- **Smallest eigenvalue is $0$**, with eigenvector $\mathbf{1} = (1, 1, \ldots, 1)^\top$ (each row of $L$ sums to zero).
- **Multiplicity of $0$** equals the number of connected components. $L$'s kernel directly detects graph disconnectivity.

Why "Laplacian"? The classical continuous Laplacian $\Delta = \sum_i \partial^2 / \partial x_i^2$ applied to a function $f$ measures how much $f$ deviates locally from the average of its neighbourhood. On a graph the analogous local operation is
$$(Lx)_i = d_i x_i - \sum_{j \in \mathcal{N}(i)} x_j = \sum_{j \in \mathcal{N}(i)} (x_i - x_j),$$
which is exactly the sum over edges incident to $i$ of how much $x_i$ differs from its neighbours. Small $(Lx)_i$ means $x$ is smooth around $i$; large means $x$ has high local variation. This makes $L$ the natural operator for defining "frequency" on a graph.


In [ ]:
# Laplacian on the 5-node example
L_small = D - A
print("Unnormalized Laplacian L = D - A on the 5-node example:")
print(L_small)
print("\nRow sums (should all be 0):", L_small.sum(axis=1))

eigvals_small, eigvecs_small = np.linalg.eigh(L_small)
print(f"\nEigenvalues (ascending): {np.round(eigvals_small, 4)}")
print(f"Smallest eigenvalue ~= 0, corresponding eigenvector is constant:")
print(np.round(eigvecs_small[:, 0], 4))
print(f"(expected: 1/sqrt(N) ~= {1/np.sqrt(N):.4f})")


### 3.2 Normalized Laplacian

The **symmetrically normalized Laplacian** is
$$L' = I_N - D^{-1/2} A D^{-1/2}.$$

Properties:

- Still symmetric and positive semi-definite.
- **Eigenvalues lie in $[0, 2]$**, with $\lambda_0 = 0$.
- Better-conditioned than $L$ for numerical work: scale is independent of node degrees.
- The matrix $D^{-1/2} A D^{-1/2}$ is exactly the **no-self-loop version** of the GCN propagation operator. Adding self-loops gives the "renormalization trick" in §3.10: $D^{-1/2} A D^{-1/2} \to \hat{D}^{-1/2} \hat{A} \hat{D}^{-1/2}$, and the corresponding Laplacian becomes $I_N - \hat{D}^{-1/2} \hat{A} \hat{D}^{-1/2}$.


In [ ]:
# Compute L' on Karate Club
d_kc          = A_kc.sum(axis=1)
D_kc_inv_sqrt = np.diag(d_kc ** -0.5)
L_sym_kc      = np.eye(n) - D_kc_inv_sqrt @ A_kc @ D_kc_inv_sqrt
eigvals_kc, eigvecs_kc = np.linalg.eigh(L_sym_kc)

print(f"Karate Club L' - eigenvalue range: [{eigvals_kc[0]:.4f}, {eigvals_kc[-1]:.4f}]")
print(f"Number of zero eigenvalues: {int(np.isclose(eigvals_kc, 0, atol=1e-8).sum())}"
      f"   (matches #components = 1)")

fig, ax = plt.subplots(figsize=(7, 3))
ax.stem(range(len(eigvals_kc)), eigvals_kc, basefmt=' ')
ax.set_xlabel('eigenvalue index')
ax.set_ylabel('lambda')
ax.set_title("Karate Club: eigenvalues of the normalized Laplacian L'")
ax.axhline(2.0, color='red', linestyle='--', lw=0.7, label='lambda = 2 upper bound')
ax.legend()
plt.tight_layout(); plt.show()


### 3.3 The convolution theorem

For continuous functions $f, h: \mathbb{R} \to \mathbb{C}$, the **convolution** is
$$(f * h)(t) = \int_{-\infty}^{\infty} f(\tau) h(t - \tau) \, d\tau,$$
and the **Fourier transform** is
$$\hat{f}(\omega) = \int_{-\infty}^{\infty} f(t) e^{-2\pi i \omega t} \, dt.$$

The **convolution theorem** says that the Fourier transform turns convolution into pointwise multiplication:
$$\widehat{f * h}(\omega) = \hat{f}(\omega) \cdot \hat{h}(\omega).$$

Equivalently, convolution can be computed by (a) Fourier-transforming both signals, (b) multiplying them pointwise, (c) inverse-transforming:
$$f * h = \mathcal{F}^{-1}\!\bigl[\mathcal{F}[f] \cdot \mathcal{F}[h]\bigr].$$

This is the key observation that extends convolution to graphs. The spatial definition of convolution (slide a kernel over the input) requires shift-invariance, which graphs do not have. The spectral definition only requires a notion of frequency domain, which we construct from the eigenvectors of the Laplacian.


### 3.4 Classical Fourier transform — the missing link

The classical Fourier transform expands a signal $f(t)$ in the basis of complex exponentials:
$$f(t) = \int \hat{f}(\omega) e^{2\pi i \omega t} d\omega, \qquad \hat{f}(\omega) = \int f(t) e^{-2\pi i \omega t} dt.$$

**Key observation** that generalises to graphs: the complex exponentials $\{e^{2\pi i \omega t}\}_\omega$ are **eigenfunctions** of the 1-D Laplace operator $\Delta = d^2/dt^2$:
$$\Delta \bigl(e^{2\pi i \omega t}\bigr) = -(2\pi \omega)^2 \, e^{2\pi i \omega t}.$$

So "taking the Fourier transform" can be rephrased as "expanding a signal in the eigenbasis of the Laplace operator". The coefficient $\hat f(\omega)$ measures how much the basis function at frequency $\omega$ contributes to $f$.

This rephrasing lets us transfer Fourier analysis to graphs: replace the continuous $\Delta$ with the discrete graph Laplacian $L$ and expand functions on nodes in $L$'s eigenbasis. That gives the **graph Fourier transform**.


### 3.5 Graph Fourier transform

Since $L$ (and $L'$) is symmetric and PSD it admits a full orthonormal eigendecomposition
$$L = U \Lambda U^\top, \qquad U = [u_0 \mid u_1 \mid \ldots \mid u_{N-1}], \qquad \Lambda = \operatorname{diag}(\lambda_0, \ldots, \lambda_{N-1}),$$
where $U$ is orthogonal ($U^\top U = I_N$). The columns $u_l$ are the **graph Fourier basis functions**, and $\lambda_l$ are **graph frequencies**: $u_0$ is constant (zero frequency), and higher-$\lambda$ eigenvectors oscillate more rapidly across the graph.

For a signal $f \in \mathbb{R}^N$ (one scalar per node), the **graph Fourier transform** is
$$\hat{f}(\lambda_l) = \langle f, u_l \rangle = \sum_{i=1}^N f_i u_l(i), \qquad \text{or in matrix form} \quad \hat{f} = U^\top f.$$

The **inverse graph Fourier transform** reconstructs $f$ from its spectral coefficients:
$$f_i = \sum_{l=0}^{N-1} \hat{f}(\lambda_l) u_l(i), \qquad f = U \hat{f}.$$

Because $U$ is orthogonal, $U U^\top = I_N$, so the forward–inverse round trip is exact.

**High-frequency vs low-frequency graph signals.** An eigenvector $u_l$ with small $\lambda_l$ varies slowly across edges (neighbouring nodes have similar values). One with large $\lambda_l$ flips sign across many edges. A graph signal $f$ with spectrum concentrated at low $\lambda$ is *smooth* (looks like community membership); one concentrated at high $\lambda$ is *noisy* or oscillatory.


In [ ]:
# Visualize the first few graph Fourier basis functions on Karate Club
# (eigvals_kc, eigvecs_kc were computed above; eigvecs_kc[:, k] is u_k)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for k, ax in zip([0, 1, 2, 10], axes):
    uk = eigvecs_kc[:, k]
    node_col_k = [plt.cm.coolwarm((uk[i] - uk.min()) / (uk.max() - uk.min() + 1e-12))
                  for i in range(n)]
    nx.draw_networkx(G_kc, pos=pos, node_color=node_col_k, node_size=220,
                     font_size=6, edge_color='#dddddd', with_labels=False, ax=ax)
    ax.set_title(f'u_{k}   lambda = {eigvals_kc[k]:.3f}')
    ax.axis('off')

plt.suptitle('Graph Fourier basis on Karate Club  (red = positive, blue = negative)',
             y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# Project the community-label signal to the spectral domain
#   f_i = +1 if Mr. Hi, -1 if Officer
f = np.array([+1.0 if club[i] == 'Mr. Hi' else -1.0 for i in G_kc.nodes()])

f_hat = eigvecs_kc.T @ f         # forward graph Fourier transform
f_rec = eigvecs_kc   @ f_hat     # inverse graph Fourier transform

print(f"Reconstruction error  |f - U U^T f|_inf = {np.abs(f - f_rec).max():.2e}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].stem(range(n), f_hat, basefmt=' ')
axes[0].set_xlabel('eigenvalue index l')
axes[0].set_ylabel('f_hat(lambda_l)')
axes[0].set_title('Spectrum of the community-label signal')

axes[1].bar(range(n), np.abs(f_hat), color='steelblue')
axes[1].set_xlabel('l'); axes[1].set_ylabel('|f_hat(lambda_l)|')
axes[1].set_title('Magnitudes -- label signal is mostly low-frequency')
plt.tight_layout(); plt.show()


### 3.6 Spectral graph convolution

Apply the convolution theorem on the graph: given a signal $f \in \mathbb{R}^N$ and a convolutional filter $h \in \mathbb{R}^N$, define the graph convolution by going through the graph Fourier domain:

$$(f *_G h) = U\Bigl[(U^\top f) \odot (U^\top h)\Bigr],$$

where $\odot$ is pointwise (Hadamard) multiplication.

Let $\hat h = U^\top h \in \mathbb{R}^N$ be the spectral representation of the filter. The convolution can then be written as a matrix:

$$f *_G h = U \operatorname{diag}(\hat h)\, U^\top f.$$

The **spectral filter** is parametrised directly in the frequency domain: instead of learning a kernel in node space, we learn a function $g_\theta(\lambda)$ that re-weights each frequency. Plugging this in:
$$y = g_\theta(L) f = U\, g_\theta(\Lambda)\, U^\top f, \qquad g_\theta(\Lambda) = \operatorname{diag}\!\bigl(g_\theta(\lambda_0), \ldots, g_\theta(\lambda_{N-1})\bigr).$$

Different choices of the functional form for $g_\theta$ give different spectral GNNs.


### 3.7 Spectral GCN Version 1.0 — Bruna et al. 2013

*Spectral Networks and Locally Connected Networks on Graphs* (Bruna, Zaremba, Szlam, LeCun 2013) makes the spectral filter **fully learnable**:

$$g_\theta(\Lambda) = \operatorname{diag}(\theta_0, \theta_1, \ldots, \theta_{N-1}), \qquad \theta \in \mathbb{R}^N.$$

One learnable scalar per eigenvalue. The forward pass of a single spectral filter is
$$y = \sigma\!\bigl(U\, g_\theta(\Lambda)\, U^\top x\bigr).$$

**Properties.**

- Most general linear, equivariant-in-the-eigenbasis filter — can represent any spectral response.
- **Parameters per filter = $N$.** Parameter count scales with graph size, so learnt weights cannot transfer between graphs of different size.
- **Not spatially localized.** A single $\theta_l$ couples every pair of nodes through the outer product $u_l u_l^\top$; there is no notion of "$k$-hop receptive field".
- **Requires eigendecomposition.** Computing $U, \Lambda$ costs $O(N^3)$ up front and the forward pass is $O(N^2)$ per filter — intractable beyond a few thousand nodes.

These issues motivate Version 2.0.


**Note on supervision.** The spatial GCN in Part III could train from only 2 labelled nodes because the Kipf & Welling "renormalization trick" and symmetric normalization impose a strong spatial-smoothness prior. Vanilla spectral filters have **no such locality prior** — they operate globally in the eigenbasis — so they need more labels to identify a meaningful frequency response. Below we use a richer supervision set (6 labels per community) to give the spectral models a fair chance.


In [ ]:
# Spectral GCN V1.0 -- from scratch on a 24-node stochastic block model.
torch.manual_seed(0)
rng_v1 = np.random.default_rng(0)

N_small   = 24
block_ids = np.array([0]*12 + [1]*12)
p_intra, p_inter = 0.45, 0.06
A_s = np.zeros((N_small, N_small))
for i in range(N_small):
    for j in range(i + 1, N_small):
        p = p_intra if block_ids[i] == block_ids[j] else p_inter
        if rng_v1.uniform() < p:
            A_s[i, j] = A_s[j, i] = 1

d_s          = A_s.sum(1)
D_s_inv_sqrt = np.diag(np.where(d_s > 0, d_s ** -0.5, 0))
L_sym_s      = np.eye(N_small) - D_s_inv_sqrt @ A_s @ D_s_inv_sqrt
eigvals_s, eigvecs_s = np.linalg.eigh(L_sym_s)

U_s = torch.tensor(eigvecs_s, dtype=torch.float32)
X_s = torch.eye(N_small, dtype=torch.float32)     # one-hot id features


class SpectralGCNv1(nn.Module):
    '''Bruna et al. 2013 spectral GCN with fully-learnable diagonal filter.

    Per layer: theta has shape (F_in, F_out, N) -- one scalar per
    (input channel, output channel, eigenvalue).
    '''

    def __init__(self, N, in_features, hidden, num_classes, U):
        super().__init__()
        self.register_buffer('U',  U)
        self.register_buffer('Ut', U.t())
        self.theta1 = nn.Parameter(torch.randn(in_features, hidden,      N) * 0.1)
        self.theta2 = nn.Parameter(torch.randn(hidden,      num_classes, N) * 0.1)

    def _spec_conv(self, X, theta):
        X_hat   = self.Ut @ X                                   # (N, F_in)
        out_hat = torch.einsum('nf,fon->no', X_hat, theta)      # (N, F_out)
        return self.U @ out_hat

    def forward(self, X):
        H = F.relu(self._spec_conv(X, self.theta1))
        Z = self._spec_conv(H, self.theta2)
        return Z


y_s    = torch.tensor(block_ids)
mask_s = torch.zeros(N_small, dtype=torch.bool)
# 6 labels per community (12 labelled nodes out of 24)
mask_s[[0, 1, 2, 3, 4, 5,  12, 13, 14, 15, 16, 17]] = True

torch.manual_seed(0)
model_v1 = SpectralGCNv1(N_small, N_small, 8, 2, U_s)
opt_v1   = torch.optim.Adam(model_v1.parameters(), lr=0.05, weight_decay=5e-3)
for _ in range(500):
    opt_v1.zero_grad()
    logits = model_v1(X_s)
    F.cross_entropy(logits[mask_s], y_s[mask_s]).backward()
    opt_v1.step()

with torch.no_grad():
    preds_v1 = model_v1(X_s).argmax(1)
    acc_v1   = (preds_v1 == y_s).float().mean().item()
    unlab    = ~mask_s
    acc_v1_u = (preds_v1[unlab] == y_s[unlab]).float().mean().item()

n_params_v1 = sum(p.numel() for p in model_v1.parameters())
print(f"Spectral GCN V1.0 parameters        : {n_params_v1}  (scales with N)")
print(f"Spectral GCN V1.0 accuracy all nodes: {acc_v1:.1%}")
print(f"Spectral GCN V1.0 accuracy unlabeled: {acc_v1_u:.1%}")


### 3.8 Spectral GCN Version 2.0 — ChebNet (Defferrard et al. 2016)

*Convolutional Neural Networks on Graphs with Fast Localized Spectral Filtering* (Defferrard, Bresson, Vandergheynst 2016) constrains $g_\theta$ to be a **polynomial** in the Laplacian eigenvalues:

$$g_\theta(\Lambda) = \sum_{j=0}^K \alpha_j \Lambda^j, \qquad \alpha = (\alpha_0, \ldots, \alpha_K) \in \mathbb{R}^{K+1}.$$

The magic: because $U \Lambda^j U^\top = (U \Lambda U^\top)^j = L^j$ (using $U^\top U = I$), we get

$$U g_\theta(\Lambda) U^\top = \sum_{j=0}^K \alpha_j L^j,$$

which contains **no eigendecomposition** at all. The forward pass becomes
$$y = \sigma\!\left(\sum_{j=0}^K \alpha_j L^j x\right),$$
evaluated entirely in node space through repeated sparse matrix–vector products.

**Properties.**

- **$K + 1$ parameters** per filter, independent of $N$. Transferable between graphs.
- **Spatial localization.** $L^j$ only couples nodes within graph distance $j$ (a direct consequence of $L$ being zero on non-edges). So a ChebNet layer with order $K$ has **exactly a $K$-hop receptive field** — the parameter $K$ directly controls locality.
- **Complexity**: $O(K \cdot |\mathcal{E}|)$ per layer using sparse multiplication.
- **No eigendecomposition**, so it scales to graphs with millions of nodes.


### 3.9 Chebyshev polynomial recursion

Computing $L^j$ explicitly is numerically poor (eigenvalues of $L'$ range in $[0, 2]$, so $L^j$'s spectrum amplifies powers of values near 2). Defferrard et al. use the **Chebyshev polynomials of the first kind** as a better-conditioned polynomial basis:

$$T_0(x) = 1, \quad T_1(x) = x, \quad T_j(x) = 2x\, T_{j-1}(x) - T_{j-2}(x).$$

Chebyshev polynomials are bounded on $[-1, 1]$: $|T_j(x)| \le 1$. We therefore **rescale** the Laplacian spectrum from $[0, \lambda_\max]$ into $[-1, 1]$:
$$\tilde{L} = \frac{2}{\lambda_\max} L' - I_N, \qquad \tilde{\Lambda} = \frac{2}{\lambda_\max} \Lambda - I_N.$$

The filter is expressed in the Chebyshev basis:
$$g_{\theta'}(\tilde{\Lambda}) = \sum_{j=0}^K \theta_j' \, T_j(\tilde{\Lambda}),$$
and applying it to a signal $x$ uses the **same three-term recurrence**, now acting on vectors:
$$
\begin{aligned}
T_0(\tilde L)\, x &= x, \\
T_1(\tilde L)\, x &= \tilde L x, \\
T_j(\tilde L)\, x &= 2 \tilde L\, T_{j-1}(\tilde L)\, x - T_{j-2}(\tilde L)\, x.
\end{aligned}
$$

Each recursion step is one sparse matrix–vector multiplication. Total cost: $O(K \cdot |\mathcal{E}|)$.


In [ ]:
class ChebConv(nn.Module):
    '''Chebyshev spectral graph convolution (Defferrard et al. 2016).

    y = sum_{j=0}^K  W_j  T_j(L_tilde) X     -- no eigendecomposition.
    '''

    def __init__(self, in_features: int, out_features: int, K: int):
        super().__init__()
        self.K = K
        self.weights = nn.ParameterList([
            nn.Parameter(torch.empty(in_features, out_features))
            for _ in range(K + 1)
        ])
        for W in self.weights:
            nn.init.xavier_uniform_(W)

    def forward(self, X: torch.Tensor, L_tilde: torch.Tensor) -> torch.Tensor:
        # T_0(L_tilde) X = X
        Tx_prev = X
        out     = Tx_prev @ self.weights[0]

        if self.K >= 1:
            Tx_curr = L_tilde @ X                             # T_1(L_tilde) X = L_tilde X
            out     = out + Tx_curr @ self.weights[1]

        for k in range(2, self.K + 1):
            Tx_next = 2 * (L_tilde @ Tx_curr) - Tx_prev       # three-term recurrence
            out     = out + Tx_next @ self.weights[k]
            Tx_prev, Tx_curr = Tx_curr, Tx_next

        return out


def build_L_tilde(A: np.ndarray) -> torch.Tensor:
    '''Rescaled normalized Laplacian  L_tilde = 2 L' / lambda_max - I.'''
    n          = A.shape[0]
    d          = A.sum(axis=1)
    D_inv_sqrt = np.diag(np.where(d > 0, d ** -0.5, 0))
    L_sym      = np.eye(n) - D_inv_sqrt @ A @ D_inv_sqrt
    lambda_max = np.linalg.eigvalsh(L_sym).max()
    L_tilde    = (2.0 / lambda_max) * L_sym - np.eye(n)
    return torch.tensor(L_tilde, dtype=torch.float32)


**Caveat on the table below.** Vanilla ChebNet does *not* have the Kipf renormalization trick; the parameter count grows rapidly with $K$ and hidden width, and training stability degrades for deep or wide filters on graphs with limited supervision. The $K=1$ result is informative; $K \ge 2$ typically requires either more labels, richer node features (not one-hot identity), or the residual / gating mechanisms mentioned in §2.2. We report all three anyway because the trend is itself instructive.


In [ ]:
# Train ChebNet on Karate Club with K = 1, 2, 3 and compare
L_tilde_kc = build_L_tilde(A_kc)


class ChebNet(nn.Module):
    def __init__(self, in_features, hidden, num_classes, K):
        super().__init__()
        self.conv1 = ChebConv(in_features, hidden,     K)
        self.conv2 = ChebConv(hidden,      num_classes, K)

    def forward(self, X, L_tilde):
        H = F.relu(self.conv1(X, L_tilde))
        Z =         self.conv2(H, L_tilde)
        return Z


# Pick 6 labelled nodes from each of the two communities (12 / 34 total)
hi_ids  = [i for i, c in enumerate(G_kc.nodes()) if club[i] == 'Mr. Hi'][:6]
off_ids = [i for i, c in enumerate(G_kc.nodes()) if club[i] == 'Officer'][:6]
mask_cheb = torch.zeros(n, dtype=torch.bool)
mask_cheb[hi_ids + off_ids] = True

print(f"Labelled nodes for ChebNet demo: {mask_cheb.nonzero().squeeze().tolist()}")
print()
print(f"ChebNet on Karate Club  ({int(mask_cheb.sum())} labelled, 500 epochs)\n")
print(f"{'K':>3}  {'#params':>10}  {'accuracy':>10}  {'receptive field':>18}")
for K in (1, 2, 3):
    torch.manual_seed(0)
    mdl  = ChebNet(in_features=n, hidden=16, num_classes=2, K=K)
    optc = torch.optim.Adam(mdl.parameters(), lr=0.01, weight_decay=5e-3)
    for _ in range(500):
        optc.zero_grad()
        F.cross_entropy(mdl(X_kc, L_tilde_kc)[mask_cheb],
                        y_kc[mask_cheb]).backward()
        optc.step()
    with torch.no_grad():
        acc_c = (mdl(X_kc, L_tilde_kc).argmax(1) == y_kc).float().mean().item()
    n_p = sum(p.numel() for p in mdl.parameters())
    print(f"{K:>3}  {n_p:>10}  {acc_c:>9.1%}  {K*2:>15}-hop")


**Receptive-field verification.** The $K$-hop locality claim can be checked directly: if we drop a unit impulse at one node and pass it through a randomly-initialised `ChebConv(K=k)` layer, the output should be non-zero only within graph distance $k$ of the impulse node. The cell below verifies this on the 5-node example.


In [ ]:
# Verify that ChebConv(K=k) has exactly k-hop receptive field
node_source = 2  # node 'c'

# Graph distances from the source
dist = dict(nx.single_source_shortest_path_length(
    nx.from_numpy_array(A), node_source))
print("Graph distances from node c (index 2):", dist)
print()

L_tilde_small = build_L_tilde(A)
impulse       = torch.zeros(N, 1)
impulse[node_source, 0] = 1.0

for K in (0, 1, 2, 3):
    torch.manual_seed(0)
    layer   = ChebConv(in_features=1, out_features=1, K=K)
    with torch.no_grad():
        out = layer(impulse, L_tilde_small).squeeze().numpy()
    nonzero = [i for i in range(N) if abs(out[i]) > 1e-6]
    reached = sorted(nonzero)
    max_hop = max(dist[i] for i in reached) if reached else -1
    print(f"K={K}: non-zero at nodes {reached}   "
          f"(max graph distance from source = {max_hop})")


### 3.10 Bridge — from ChebNet K=1 to Kipf & Welling spatial GCN

The Kipf & Welling spatial rule is a *special case* of ChebNet. Setting $K = 1$ in the Chebyshev expansion:
$$y = \theta_0' T_0(\tilde{L})\, x + \theta_1' T_1(\tilde{L})\, x = \theta_0' x + \theta_1' \tilde{L}\, x.$$

**Step 1 — assume $\lambda_\max \approx 2$.** The normalized Laplacian has spectrum in $[0, 2]$, with the upper bound attained by bipartite graphs. Approximating $\lambda_\max = 2$:
$$\tilde{L} = \frac{2}{\lambda_\max} L' - I_N = L' - I_N = \bigl(I_N - D^{-1/2} A D^{-1/2}\bigr) - I_N = -D^{-1/2} A D^{-1/2}.$$

Substituting:
$$y = \theta_0' x - \theta_1' D^{-1/2} A D^{-1/2}\, x.$$

**Step 2 — tie parameters.** Kipf & Welling set $\theta_0' = -\theta_1' \triangleq \theta$, reducing the filter to one free parameter:
$$y = \theta \bigl(I_N + D^{-1/2} A D^{-1/2}\bigr) x.$$

**Step 3 — renormalization trick.** The matrix $I_N + D^{-1/2} A D^{-1/2}$ has eigenvalues in $[0, 2]$; stacking it repeatedly causes exploding / vanishing activations. Kipf & Welling replace it with the **self-loop-augmented symmetric normalization**:
$$I_N + D^{-1/2} A D^{-1/2} \;\longrightarrow\; \hat{D}^{-1/2} \hat{A} \hat{D}^{-1/2}, \qquad \hat{A} = A + I_N.$$

This is not an algebraic equality — it is a **stability fix** that plays the same structural role (mixes self and neighbours with degree normalization) but keeps the spectrum better bounded.

**Step 4 — generalize to multi-channel features.** At the single-channel level $\theta$ is a scalar. For input features $H^{(l)} \in \mathbb{R}^{N \times F_l}$ the scalar becomes a weight matrix $W^{(l)} \in \mathbb{R}^{F_l \times F_{l+1}}$, giving:
$$H^{(l+1)} = \sigma\!\bigl(\hat{D}^{-1/2} \hat{A} \hat{D}^{-1/2}\, H^{(l)}\, W^{(l)}\bigr),$$
precisely the spatial GCN layer of Part III.

**Summary of the derivation chain:**
$$\underbrace{U g_\theta(\Lambda) U^\top}_{\text{V1.0 Bruna}}
\;\longrightarrow\;
\underbrace{\sum_{j=0}^K \theta_j' T_j(\tilde L)}_{\text{V2.0 ChebNet}}
\;\xrightarrow{K=1,\; \theta_0 = -\theta_1,\; \text{renormalize}}\;
\underbrace{\hat{D}^{-1/2} \hat{A} \hat{D}^{-1/2}\, H W}_{\text{Kipf-Welling spatial GCN}}.$$


In [ ]:
# Numerical verification of the bridge:
# ChebNet(K=1) with tied parameters under the renormalization trick
# is equivalent to a spatial GCN layer.

torch.manual_seed(0)
H0 = torch.randn(n, 4)
W  = torch.randn(4, 2)

A_hat_norm_t = build_A_hat_norm(A_kc)

# (a) Spatial GCN one linear layer
spatial_out = A_hat_norm_t @ H0 @ W

# (b) ChebNet K=1 with theta_0 = -theta_1 = W and with the renormalization trick
#     (meaning: use A_hat_norm in place of  I_N + D^{-1/2} A D^{-1/2})
class ChebK1Tied(nn.Module):
    def __init__(self, W):
        super().__init__()
        self.W = nn.Parameter(W.clone())
    def forward(self, H, A_hat_norm):
        return A_hat_norm @ H @ self.W

cheb_out = ChebK1Tied(W)(H0, A_hat_norm_t)

diff = (spatial_out - cheb_out).abs().max().item()
print(f"max |spatial - ChebK1-tied-renormalized| = {diff:.2e}")
print("-> spatial GCN layer = ChebNet(K=1) under parameter tying + renormalization trick.")


---
## Part V · Prediction Tasks Beyond Node Classification

The same `GCNLayer` and the same learnt node embeddings can be reused for three task levels. Part III covered node-level classification; this part adds graph-level and edge-level tasks.

### 4.1 Graph classification with global readout

Task: given an entire graph, output a single class label (e.g., "is this molecule toxic?"). The architecture adds a **readout** layer that collapses node embeddings into a graph embedding:
$$h_\mathcal{G} = \operatorname{readout}\bigl\{ h_i : i \in \mathcal{V} \bigr\}, \qquad p(y|\mathcal{G}) = \operatorname{softmax}(W_\text{out}\, h_\mathcal{G}).$$

Common readout functions: **mean**, **sum**, **max**, or attention-weighted pooling. Mean and sum preserve permutation invariance. Kipf & Welling's original demo uses mean pooling.

We distinguish two families of synthetic random graphs:

- **Erdős–Rényi** $G(n, p)$: each possible edge exists independently with probability $p$. Roughly binomial degree distribution.
- **Barabási–Albert** (preferential attachment): add nodes one at a time; each new node connects to $m$ existing nodes chosen in proportion to their current degree. Produces a power-law degree distribution.

A GCN with mean readout can separate them because their structural signatures (degree distribution, clustering) differ.


In [ ]:
import random as _random
_random.seed(0)

def make_graph_dataset(n_graphs=60, n_nodes=20, seed=0):
    rng = np.random.default_rng(seed)
    graphs, labels = [], []
    for _ in range(n_graphs // 2):
        g0 = nx.erdos_renyi_graph(n_nodes, 0.15,
                                  seed=int(rng.integers(1_000_000)))
        graphs.append(g0); labels.append(0)
        g1 = nx.barabasi_albert_graph(n_nodes, 2,
                                      seed=int(rng.integers(1_000_000)))
        graphs.append(g1); labels.append(1)
    return graphs, labels


def graph_to_tensors(G):
    A_g = nx.to_numpy_array(G)
    X_g = torch.eye(G.number_of_nodes(), dtype=torch.float32)
    return X_g, build_A_hat_norm(A_g)


train_graphs, train_labels = make_graph_dataset(60, n_nodes=20, seed=0)
test_graphs,  test_labels  = make_graph_dataset(20, n_nodes=20, seed=99)


class GraphGCN(nn.Module):
    '''GCN encoder + mean readout + linear classifier.'''

    def __init__(self, in_f, hidden, num_classes):
        super().__init__()
        self.gcn = GCNLayer(in_f, hidden)
        self.clf = nn.Linear(hidden, num_classes)

    def forward(self, X_g, A_g_norm):
        H  = self.gcn(X_g, A_g_norm)
        hG = H.mean(dim=0)
        return self.clf(hG)


torch.manual_seed(0)
gmodel = GraphGCN(in_f=20, hidden=32, num_classes=2)
gopt   = torch.optim.Adam(gmodel.parameters(), lr=0.01)

for epoch in range(60):
    gmodel.train()
    total = 0.0
    for G_, lbl in zip(train_graphs, train_labels):
        X_g, A_g = graph_to_tensors(G_)
        gopt.zero_grad()
        logit = gmodel(X_g, A_g).unsqueeze(0)
        loss  = F.cross_entropy(logit, torch.tensor([lbl]))
        loss.backward(); gopt.step()
        total += loss.item()
    if (epoch + 1) % 20 == 0:
        print(f"epoch {epoch+1:3d}  train loss {total / len(train_graphs):.4f}")

gmodel.eval()
correct = 0
with torch.no_grad():
    for G_, lbl in zip(test_graphs, test_labels):
        X_g, A_g = graph_to_tensors(G_)
        pred = gmodel(X_g, A_g).argmax().item()
        correct += int(pred == lbl)
print(f"\nTest accuracy (Erdos-Renyi vs Barabasi-Albert, 20 graphs): "
      f"{correct / len(test_graphs):.1%}")


### 4.2 Link prediction with dot-product scoring

Task: given a graph with some edges observed and some hidden, predict the missing edges. The architecture learns node embeddings and scores a candidate edge $(i, j)$ by the inner product
$$s(i, j) = \langle h_i, h_j \rangle.$$
This is the minimum-viable **graph auto-encoder** from Kipf & Welling (NIPS BDL 2016).

Training procedure:

1. Remove a random held-out subset of edges (the positive test set).
2. Train a GCN on the pruned graph using the semi-supervised node-classification loss (or any auxiliary task that yields useful embeddings).
3. At test time, score each held-out edge against a matched set of random non-edges. Report the pairwise ranking AUC — the fraction of (positive, negative) pairs for which the positive edge scores higher.


In [ ]:
# Link prediction on Karate Club
_random.seed(42)

all_edges = list(G_kc.edges())
_random.shuffle(all_edges)
n_test    = max(1, len(all_edges) // 10)
test_pos  = all_edges[:n_test]
train_e   = all_edges[n_test:]

G_pruned = nx.Graph()
G_pruned.add_nodes_from(G_kc.nodes())
G_pruned.add_edges_from(train_e)
A_pruned      = nx.to_numpy_array(G_pruned)
A_pruned_norm = build_A_hat_norm(A_pruned)

# Train a GCN on the pruned graph using the community labels
torch.manual_seed(42)
lp_model = GCN(in_features=n, hidden=16, num_classes=2)
lp_opt   = torch.optim.Adam(lp_model.parameters(), lr=0.01, weight_decay=5e-4)
for _ in range(300):
    lp_opt.zero_grad()
    F.cross_entropy(lp_model(X_kc, A_pruned_norm)[mask_kc], y_kc[mask_kc]).backward()
    lp_opt.step()

with torch.no_grad():
    emb = lp_model.embed(X_kc, A_pruned_norm)      # (34, 16)

def score(u, v):
    return (emb[u] @ emb[v]).item()

non_edges = list(nx.non_edges(G_kc))
_random.shuffle(non_edges)
test_neg  = non_edges[:n_test]

pos_scores = [score(u, v) for u, v in test_pos]
neg_scores = [score(u, v) for u, v in test_neg]

from itertools import product as _product
auc = (sum(ps > ns for ps, ns in _product(pos_scores, neg_scores))
       / (len(pos_scores) * len(neg_scores)))

print(f"held-out edges        : {n_test}")
print(f"mean score (real)     : {np.mean(pos_scores):.4f}")
print(f"mean score (non-edge) : {np.mean(neg_scores):.4f}")
print(f"pairwise AUC          : {auc:.3f}   (random = 0.5)")

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(pos_scores, bins=10, alpha=0.7, color='#5B8DB8', label='real edges')
ax.hist(neg_scores, bins=10, alpha=0.7, color='#E07B54', label='non-edges')
ax.set_xlabel('dot-product score'); ax.set_ylabel('count')
ax.set_title('Link prediction score distributions')
ax.legend(); plt.tight_layout(); plt.show()


---
## Part VI · Claude Code and MCP (Model Context Protocol)

MCP is the protocol layer that lets Claude Code interact with external systems (filesystems, databases, APIs, GitHub, Slack, etc.) through a single standard interface. It is complementary to Skills, which shape Claude's internal behaviour: **MCP extends what Claude can reach externally; Skills shape how Claude behaves internally**.


### 5.1 What Is MCP?

**Model Context Protocol (MCP)** is an open standard for connecting AI applications to external systems. The official analogy:

> *"MCP is like USB-C for AI applications."*
> Just as USB-C provides a single standardised port to connect any peripheral device, MCP provides a single standardised protocol to connect any AI application to any external tool or data source.

**The problem MCP solves.** Before MCP, every AI application needed custom, one-off integrations for each external tool — a different connector for GitHub, another for Notion, another for a database — none of them compatible. MCP replaces that fragmented landscape with one universal protocol.

**What MCP enables.** With the right set of MCP servers connected, Claude Code can:

- Read and write files on your filesystem with proper access controls.
- Query a PostgreSQL database using natural language.
- Create GitHub issues, comment on PRs, list commits.
- Search Slack, Notion, or the open web.
- Fetch TensorBoard scalars, list Hugging Face models, query Weights & Biases runs.

**Architecture** (one sentence each):

- **MCP client** — Claude Code itself (or any other MCP-compatible host).
- **MCP server** — one process per external system, exposing a set of primitives (see §5.3).
- **Transport** — the wire protocol between client and server. Two supported transports: `stdio` (server runs as a local subprocess) and `http` (server runs on a remote machine, reached via HTTPS).


### 5.2 MCP vs. Skills — Two Complementary Mechanisms

A common point of confusion is how MCP relates to the Skills we covered in Week 5. The key distinction is:

> **MCP is how Claude Code *gets* information and capabilities from the outside world.**
> **Skills are how Claude Code *handles* and *processes* information for your project.**

Think of it this way: Skills teach Claude *your workflow* — how you like code structured, what experiments to run, what format to use for reports. MCP gives Claude *new senses and tools* — the ability to look at a GitHub issue, query a database, or browse a website.

| Dimension | MCP | Skills |
|-----------|-----|--------|
| **Core purpose** | Connect to external systems (data & actions) | Customise Claude's behaviour for your project |
| **What it provides** | New tools (actions), resources (data), prompts (templates) | Instructions, workflow automation, project context |
| **Where it lives** | `.mcp.json` or `~/.claude.json` — server config | `.claude/skills/<name>/SKILL.md` — markdown instructions |
| **Execution** | Communicates with an external server process via JSON-RPC | Runs entirely inside Claude Code's context window |
| **Analogy** | External sensors and actuators | The agent's internal training and working memory |
| **Example** | Query a live PostgreSQL database for row counts | Automatically format results as a LaTeX table for your paper |
| **Context scope** | Fetches fresh data on each invocation | Can reference full conversation history |

**They work together.** A typical deep-learning research workflow might:
1. Use the **Filesystem MCP** to read raw experiment logs (MCP fetches the data)
2. Invoke a custom `/summarise-run` **Skill** to parse and structure the results (Skill processes the data)
3. Use the **GitHub MCP** to open a pull request with the summary (MCP takes an action)

Neither replaces the other — MCP expands what Claude *can reach*, Skills shape *how it behaves*.

### 5.3 What an MCP Server Provides

Every MCP server exposes up to three types of capability. Understanding these three primitives makes it easy to predict what any given server can do.

#### Tools — Executable Actions

Tools are functions the server exposes that Claude can call. When you ask Claude to "create a GitHub issue" or "query the database for rows where loss < 0.1", it is calling a **tool** on an MCP server.

Each tool has:
- A **name** (e.g., `create_issue`, `query_database`)
- A **description** Claude reads to decide when to invoke it
- An **input schema** specifying what parameters to pass

Tools are invoked **automatically** — Claude decides on its own when a tool is the right way to answer your request. You do not have to explicitly say "use the GitHub MCP tool".

#### Resources — Data Sources

Resources are read-only data items the server can expose. You reference them using `@server:protocol://path` syntax, or Claude fetches them automatically when relevant.

Examples:
- `@github:issue://456` — full text and metadata of GitHub issue #456
- `@postgres:schema://public` — schema of the public database schema
- `@filesystem:file:///home/user/project/results.csv` — a local file served securely

#### Prompts — Reusable Templates

Prompts are workflow templates a server ships alongside its tools. They appear in Claude Code as `/mcp__<server>__<prompt>` slash commands.

```
/mcp__github__pr_review 123          # template for reviewing PR #123
/mcp__db__query_builder              # guided SQL query builder
```

| Primitive | What it does | How you invoke it | Example |
|-----------|-------------|-------------------|---------|
| **Tool** | Performs an action or computation | Auto-invoked by Claude on request | `create_issue`, `run_query` |
| **Resource** | Provides read-only data or context | `@server:protocol://path` | `@github:issue://456` |
| **Prompt** | Pre-packaged workflow template | `/mcp__server__prompt_name` | `/mcp__github__pr_review` |

### 5.4 How to Use MCP in Claude Code

#### Adding an MCP Server

The core command is:

```bash
claude mcp add --transport <type> <name> <connection-params>
```

There are two common transport types:

**HTTP (remote server — most common for third-party services):**
```bash
claude mcp add --transport http github https://api.githubcopilot.com/mcp/
claude mcp add --transport http sentry https://mcp.sentry.dev/mcp
claude mcp add --transport http notion https://mcp.notion.com/mcp
```

**Stdio (local process — for servers that run on your machine):**
```bash
# Note: double-dash separates Claude's flags from the server command
claude mcp add --transport stdio filesystem -- npx -y @modelcontextprotocol/server-filesystem /home/user/project
claude mcp add --transport stdio --env AIRTABLE_API_KEY=mykey airtable -- npx -y airtable-mcp-server
```

#### Scopes — Who Can See This Server?

Every MCP server is registered at one of three scopes:

| Scope | Config file | Visibility | When to use |
|-------|-------------|------------|-------------|
| `local` (default) | `~/.claude.json` under project path | Only you, only in this project | Personal servers, secrets, experimental |
| `project` | `.mcp.json` in project root (commit to git) | Entire team | Shared team tools |
| `user` | `~/.claude.json` in home dir | You, across all projects | Personal utilities you use everywhere |

```bash
# Explicitly set scope
claude mcp add --transport http --scope project paypal https://mcp.paypal.com/mcp
claude mcp add --transport http --scope user hubspot  https://mcp.hubspot.com/anthropic
```

#### Managing Installed Servers

```bash
claude mcp list            # list all configured servers
claude mcp get github      # inspect a specific server
claude mcp remove github   # remove a server
```

#### Invoking MCP Inside Claude Code

Once a server is added, you don't need special syntax for tools — Claude calls them automatically:

```
> Summarise all open issues labelled "bug" in my repository
# Claude automatically calls the GitHub MCP list_issues tool

> What tables exist in the production database?
# Claude automatically calls the PostgreSQL MCP list_tables tool
```

For resources and prompts, use the explicit syntax:
```
> Analyse @github:issue://789 and suggest a fix
> /mcp__github__pr_review 42
```

Check server status at any time inside Claude Code with `/mcp`.

### 5.5 Popular MCP Servers and What They Do

The official MCP registry at [registry.modelcontextprotocol.io](https://registry.modelcontextprotocol.io) lists hundreds of servers. Below are the most widely used, grouped by category.

#### Developer Tools

| Server | What it does | Example prompts |
|--------|-------------|----------------|
| **GitHub** | Manage repositories, list/create issues and PRs, review code, search commits | *"Open an issue for the failing test in train.py"*; *"List all PRs mentioning 'dropout'"* |
| **Git** (local) | Read local git history, diff, branch, commit | *"Summarise what changed in the last 10 commits"* |
| **Sentry** | Browse error reports, stack traces, crash analytics | *"What are the top 5 errors in production this week?"* |

#### Data & Databases

| Server | What it does | Example prompts |
|--------|-------------|----------------|
| **Filesystem** (local) | Secure read/write access to local files and directories | *"Read all `.log` files under `runs/` and show me validation losses"* |
| **PostgreSQL / DBHub** (local) | Natural-language queries against any SQL database | *"How many training samples have label == 3?"* |
| **Memory** (local) | Persistent knowledge graph that survives across Claude Code sessions | *"Remember that we use AdamW with lr=1e-3 for all experiments"* |

#### Web & Research

| Server | What it does | Example prompts |
|--------|-------------|----------------|
| **Web Fetch** | Retrieve and parse any web page | *"Fetch the PyTorch docs for `nn.MultiheadAttention` and summarise the parameters"* |
| **Brave Search** | Live internet search with cited results | *"Find the latest papers on graph transformers published in 2025"* |

#### Productivity & Communication

| Server | What it does | Example prompts |
|--------|-------------|----------------|
| **Slack** | Send messages, search channels, summarise threads | *"Post a summary of today's experiment results to #ml-research"* |
| **Notion** | Read and write Notion databases and pages | *"Add a new row to our Results database with the F1 score from this run"* |
| **Airtable** | Query and update Airtable bases | *"Update the Status column to 'Done' for Experiment #42"* |

#### For Deep Learning Research Specifically

The combination most useful for this course:

```
Filesystem MCP   →  read raw experiment outputs, CSVs, checkpoints
GitHub MCP       →  track issues, submit code reviews, log results as comments
Web Fetch MCP    →  pull up paper abstracts and PyTorch documentation on demand
Memory MCP       →  persist hyperparameter decisions and experiment notes across sessions
```

> **Security note:** MCP servers that fetch untrusted external content (web pages, third-party APIs) can expose you to **prompt injection** — adversarially crafted content designed to hijack Claude's behaviour. Always use well-known, reputable servers for production work, and review the output of any external fetch before acting on it.

### 5.6 Command quick-reference

The cell below is a terminal-only reference — the commands are all meant to be run in your shell, not inside the notebook. Leave them commented out.


In [ ]:
# ============================================================
# MCP Quick-Reference — run these in your terminal, not here
# ============================================================

# --- ADDING MCP SERVERS ---

# Add a remote HTTP server (most third-party services)
# claude mcp add --transport http github https://api.githubcopilot.com/mcp/
# claude mcp add --transport http sentry  https://mcp.sentry.dev/mcp
# claude mcp add --transport http notion  https://mcp.notion.com/mcp

# Add a local stdio server (runs as a process on your machine)
# claude mcp add --transport stdio filesystem -- \
#     npx -y @modelcontextprotocol/server-filesystem /path/to/project

# Add with environment variable (for API key auth)
# claude mcp add --transport stdio \
#     --env AIRTABLE_API_KEY=YOUR_KEY \
#     airtable -- npx -y airtable-mcp-server

# Add at project scope (shared with team via .mcp.json)
# claude mcp add --transport http --scope project sentry https://mcp.sentry.dev/mcp

# --- MANAGING SERVERS ---
# claude mcp list              # see all registered servers
# claude mcp get github        # inspect a specific server config
# claude mcp remove github     # deregister a server

# --- INSIDE CLAUDE CODE ---
# /mcp                         # view connected servers & authenticate

# Invoke a resource by reference:
# > Analyse @github:issue://789 and suggest a fix

# Invoke a prompt template:
# > /mcp__github__pr_review 42

# Everything else: just describe what you want —
# Claude will call the appropriate MCP tool automatically.
#
# Example natural-language invocations:
# > "Read the validation loss from runs/exp01/metrics.csv and plot it"
# > "What are the top bugs in my Sentry project this week?"
# > "Commit my changes with a descriptive message"
# > "Search the web for the ViT paper and give me a one-paragraph summary"

print("MCP quick-reference loaded — see comments above for terminal commands.")